# PRT565 — Cost-Sensitive Models
Tests phishing misclassification weights **1, 2, 5, 10** and measures the missed-phishing vs false-alarm trade-off.

In [ ]:
# If needed: %pip install pandas numpy scikit-learn matplotlib

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
DATA_DIR=Path('.')
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix
OUT=Path('results_cost_sensitive'); OUT.mkdir(exist_ok=True)

In [2]:
def xy(d): return d.drop(columns='label'),d['label']
train=pd.read_csv(DATA_DIR/'train.csv'); test=pd.read_csv(DATA_DIR/'test.csv'); train_s=pd.read_csv(DATA_DIR/'train_scaled.csv'); test_s=pd.read_csv(DATA_DIR/'test_scaled.csv')
Xtr,ytr=xy(train); Xte,yte=xy(test); Xtr_s,_=xy(train_s); Xte_s,_=xy(test_s)

In [3]:
def metrics(name,w,m,Xa,Xb):
 p=m.predict(Xb); prob=m.predict_proba(Xb)[:,1]; cm=confusion_matrix(yte,p,labels=[0,1])
 return {'Model':name,'Phishing Cost Weight':w,'Accuracy':accuracy_score(yte,p),'Phishing Precision':precision_score(yte,p,pos_label=0),'Phishing Recall':recall_score(yte,p,pos_label=0),'Phishing F1':f1_score(yte,p,pos_label=0),'ROC-AUC':roc_auc_score(yte,prob),'Phishing Missed (FN)':int(cm[0,1]),'Legitimate Flagged (FP)':int(cm[1,0])}
rows=[]
for w in [1,2,5,10]:
 for name,m in [('Decision Tree',DecisionTreeClassifier(random_state=42,class_weight={0:w,1:1})),('Random Forest',RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1,class_weight={0:w,1:1}))]:
  m.fit(Xtr,ytr); rows.append(metrics(name,w,m,Xtr,Xte))
 sw=np.where(ytr.to_numpy()==0,w,1.0); nb=GaussianNB(); nb.fit(Xtr_s,ytr,sample_weight=sw); rows.append(metrics('Gaussian Naive Bayes',w,nb,Xtr_s,Xte_s))
 # Weighted resampling for MLP (portable across sklearn versions)
 rng=np.random.default_rng(42+w); pr=np.where(ytr.to_numpy()==0,w,1.0); pr=pr/pr.sum(); idx=rng.choice(np.arange(len(ytr)),size=len(ytr),replace=True,p=pr) if w>1 else np.arange(len(ytr))
 mlp=MLPClassifier(hidden_layer_sizes=(128,64),max_iter=100,early_stopping=True,validation_fraction=.15,random_state=42); mlp.fit(Xtr_s.iloc[idx],ytr.iloc[idx]); rows.append(metrics('Multilayer ANN',w,mlp,Xtr_s,Xte_s))
r=pd.DataFrame(rows); display(r.round(4)); r.to_csv(OUT/'cost_sensitive_results.csv',index=False)

,Model,Phishing Cost Weight,Accuracy,Phishing Precision,Phishing Recall,Phishing F1,ROC-AUC,Phishing Missed (FN),Legitimate Flagged (FP)
0,Decision Tree,1,1.0000,1.0000,1.0000,1.0000,1.0000,0,0
1,Random Forest,1,1.0000,1.0000,1.0000,1.0000,1.0000,0,0
2,Gaussian Naive Bayes,1,0.9996,0.9993,0.9998,0.9995,0.9999,3,11
3,Multilayer ANN,1,0.9999,0.9998,1.0000,0.9999,1.0000,0,3
4,Decision Tree,2,1.0000,1.0000,1.0000,1.0000,1.0000,0,0
5,Random Forest,2,1.0000,1.0000,1.0000,1.0000,1.0000,0,0
6,Gaussian Naive Bayes,2,0.9996,0.9992,0.9998,0.9995,0.9999,3,12
7,Multilayer ANN,2,0.9999,1.0000,0.9999,0.9999,1.0000,2,0
8,Decision Tree,5,1.0000,1.0000,1.0000,1.0000,1.0000,0,0
9,Random Forest,5,1.0000,1.0000,1.0000,1.0000,1.0000,0,0


In [ ]:
for name,g in r.groupby('Model'):
 g=g.sort_values('Phishing Cost Weight'); plt.figure(figsize=(7,5)); plt.plot(g['Phishing Cost Weight'],g['Phishing Missed (FN)'],marker='o',label='Missed phishing'); plt.plot(g['Phishing Cost Weight'],g['Legitimate Flagged (FP)'],marker='o',label='False alarms'); plt.xlabel('Phishing Cost Weight'); plt.ylabel('Test cases'); plt.title(name+' — FN vs FP trade-off'); plt.legend(); plt.tight_layout(); plt.savefig(OUT/f"{name.replace(' ','_')}_tradeoff.png",dpi=200); plt.show()